In [212]:
import numpy as np
import pennylane as qml
from scipy.linalg import fractional_matrix_power
from scipy.optimize import minimize
from pennylane import numpy as np

In [213]:
m_e  = 1
m_star = 0.16
eps_0 = 1/(4*np.pi)
eps_r = 8.8
eps = eps_r * eps_0
alpha_r_znse = ((m_e*m_star)/eps_r)**2

In [214]:
alpha_dict = {
    "a_1": 17.38354739 * alpha_r_znse,
    "a_2": 3.185489246 * alpha_r_znse,
    "a_3": 0.8897299079 * alpha_r_znse,
    "a_4": 0.3037874103 * alpha_r_znse,
    "a_5": 0.1144784984 * alpha_r_znse
} # ST0-5G

In [215]:
N = len(alpha_dict)

In [216]:
def pe_H_func(n, empty_dict, dict_of_alpha):
 
    constant_pe = -m_e**2 / (2 * eps)

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "V_H_" + str(i) + str(j)
            denominator = (dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)])
            value = 1/denominator
            empty_dict[key] = value * constant_pe
    return empty_dict

In [217]:
def pe_cc_func(n, empty_dict, dict_of_alpha):
 
    constant = -(m_e**2) * ((1/eps_0) - (1/eps))

    r_cc = 0.1521 #In
   

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "V_cc_" + str(i) + str(j)
            p =  (dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)])
            w1 = 0.85355339
            t1 = 2 - (2**0.5)
            w2 = 0.14644661
            t2 = 2 + (2**0.5)
            w1t1 = (r_cc**2)*w1*np.exp(-p*(r_cc**2)*(t1**2))*t1
            w2t2 = (r_cc**2)*w2*np.exp(-p*(r_cc**2)*(t2**2))*t2
            
            value = w1t1 + w2t2 
            
            empty_dict[key] = value * constant
    return empty_dict

In [218]:
def ke_func(n, empty_dict, dict_of_alpha):
    hbar = 1  
    constant_ke = (3*(hbar**2))/(m_star)

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "T_" + str(i) + str(j)
            numerator = (dict_of_alpha['a_'+str(i)])*(dict_of_alpha['a_'+str(j)])* (np.pi)**1.5
            denominator = ((dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)]))**2.5
            value = numerator/denominator
            empty_dict[key] = value * constant_ke
    return empty_dict

In [219]:
def overlap_func(n, empty_dict, dict_of_alpha):

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "S_" + str(i) + str(j)
            denominator = ((dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)]))
            value = (np.pi/denominator)**1.5
            empty_dict[key] = value
    return empty_dict

In [220]:
def total_pe_construction(n, empty_dict,pe_H_dictionary, pe_cc_dictionary):
    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "V_" + str(i) + str(j)
            value = pe_H_dictionary["V_H_"+str(i)+str(j)]+ pe_cc_dictionary["V_cc_"+str(i)+str(j)]
            empty_dict[key] = value
    return empty_dict

In [221]:
def Hamilt_construction(H,n,ke_dictionary, pe_dictionary):
    H = np.zeros((n,n))
    for i in range(0,n):
        for j in range(0,n):
            H[i][j] = ke_dictionary['T_'+str(i+1)+str(j+1)] + pe_dictionary['V_'+str(i+1)+str(j+1)]
    
    return H

In [222]:
def overlap_matrix_construction(S,n, overlap_dict):
    S = np.zeros((n,n))
    for i in range(0,n):
        for j in range(0,n):
            S[i][j] =  overlap_dict['S_'+str(i+1)+str(j+1)]
    
    return S

In [223]:
V_H_dict = {}
V_H_dict = pe_H_func(N,V_H_dict,alpha_dict)
V_cc_dict = {}
V_cc_dict = pe_cc_func(N,V_cc_dict,alpha_dict)
V_dict = {}
V_dict = total_pe_construction(N,V_dict,V_H_dict,V_cc_dict)
T_dict = {}
T_dict = ke_func(N,T_dict,alpha_dict)
S_dict = {}
S_dict = overlap_func(N,S_dict,alpha_dict)

In [224]:
H = ''
H = Hamilt_construction(H,N,T_dict,V_dict)

In [225]:
S = ''
S = overlap_matrix_construction(S,N,S_dict)

In [226]:
nqubits = 3

In [227]:
def EfficientSU2(theta): # eigen value function
    for i in range(nqubits):
        qml.RY(theta[i], wires = i)
        qml.RZ(theta[i+nqubits], wires = i)
    for i in range(nqubits-1):
        qml.CNOT(wires = [i, i + 1])
    for i in range(nqubits):
        qml.RY(theta[i+(2*nqubits)], wires = i)
        qml.RZ(theta[i+(3*nqubits)], wires = i)
    for i in range(nqubits-1):
        qml.CNOT(wires = [i, i + 1])
    for i in range(nqubits):
        qml.RY(theta[i+(4*nqubits)], wires = i)
        qml.RZ(theta[i+(5*nqubits)], wires = i)


In [228]:
dev = qml.device("default.qubit", wires=nqubits)
@qml.qnode(dev)
def energy(param):
    EfficientSU2(param)
    return qml.expval(H)

In [229]:
@qml.qnode(dev)  # eigen vector funtion
def EfficientSU22(theta):
    for i in range(nqubits):
            qml.RY(theta[i], wires = i)
            qml.RZ(theta[i+nqubits], wires = i)
    for i in range(nqubits-1):
        qml.CNOT(wires = [i, i + 1])
    for i in range(nqubits):
        qml.RY(theta[i+(2*nqubits)], wires = i)
        qml.RZ(theta[i+(3*nqubits)], wires = i)
    for i in range(nqubits-1):
        qml.CNOT(wires = [i, i + 1])
    for i in range(nqubits):
        qml.RY(theta[i+(4*nqubits)], wires = i)
        qml.RZ(theta[i+(5*nqubits)], wires = i)
    return qml.state()

In [230]:
S_half = fractional_matrix_power(S, -0.5)
H_prime = S_half @ H @ S_half

pad_size = 8 - H_prime.shape[0]
H = np.pad(H_prime, ((0, pad_size), (0, pad_size)), mode='constant')
H[N][N] =1000
H[N+1][N+1] =1000
H[N+2][N+2] =1000


H = qml.Hermitian(H, wires=list(range(nqubits)))

In [231]:

ground_state_energy_list = []
permater_list = []
for seed in range(10):
    theta = np.array(np.random.random(6*nqubits), requires_grad=True)
    result = minimize(energy, x0=theta, tol=10e-8)
    ground_state_energy_list.append(result.fun * 27211.38)
    permater_list.append(result.x)

print('min', min(ground_state_energy_list), 'max', max(ground_state_energy_list), 'std', np.std(ground_state_energy_list))

min -27.999449497289106 max -12.289572828912863 std 7.667758908976749


In [232]:
min_i = np.argmin(ground_state_energy_list)

print(min_i)

0


In [233]:
E_0_vec = np.abs(EfficientSU22(permater_list[min_i])) #added optimised permaters to the second circuit to find the eigenvector for E0


In [234]:
E_0_vec 

tensor([2.69135740e-02, 1.32647746e-01, 3.54979221e-01, 6.18294240e-01,
        6.88027776e-01, 2.95995200e-08, 9.65013624e-08, 1.73225474e-07], requires_grad=True)